# Error-components logit

## Model

Error components induce flexible covariance across alternatives:

$$
U_{njr}=V_{nj}
+\sum_{h\in\mathcal{H}}\ell_{jh}\sigma_h z_{nrh}
+\varepsilon_{njr},\qquad z_{nrh}\sim\mathcal{N}(0,1).
$$

The unconditional probability averages the conditional logit probability over
the component draws. This example estimates two components: a common
public-transport component for TRAIN and SM, and a rail-contrast component
with opposite TRAIN and SM loadings.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, ErrorComponent, ErrorComponentsLogit, UtilitySpec
from torchdcm.datasets import make_swissmetro_like
# Use double precision and a fixed seed for stable, reproducible estimation.
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
# Use CUDA automatically when available; set device = "cpu" to force CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
# Generate synthetic choices with two normal error components.
rng = np.random.default_rng(21)
frame = make_swissmetro_like(n_obs=500, seed=21)
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)

# Move TRAIN and SM together, then contrast TRAIN against SM.
public_draw = rng.normal(scale=0.50, size=len(frame))
rail_draw = rng.normal(scale=0.30, size=len(frame))
public_loading = np.array([1.0, 1.0, 0.0])
rail_loading = np.array([1.0, -1.0, 0.0])
# Add the error-component draws to deterministic utility.
utility = (
    np.array([0.25, 0.0, 0.15])
    - 0.025 * time
    - 0.070 * cost
    + public_draw[:, None] * public_loading
    + rail_draw[:, None] * rail_loading
)
masked = np.where(available, utility, -np.inf)
probabilities = np.exp(masked - np.max(masked, axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

# Convert simulated choices and attributes into TorchDCM choice data.
data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
# Specify mean utility separately from the covariance components.
spec = UtilitySpec()
spec.utility(
    "TRAIN",
    Beta("ASC_TRAIN")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "SM",
    Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)
spec.utility(
    "CAR",
    Beta("ASC_CAR")
    + Beta("B_TIME", init=-0.02) * "time"
    + Beta("B_COST", init=-0.05) * "cost",
)


UtilitySpec(utilities=OrderedDict({'TRAIN': Expression(terms=[Term(parameter=Beta(name='ASC_TRAIN', init=0.0, fixed=False), variable=None, multiplier=1.0), Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)]), 'SM': Expression(terms=[Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)]), 'CAR': Expression(terms=[Term(parameter=Beta(name='ASC_CAR', init=0.0, fixed=False), variable=None, multiplier=1.0), Term(parameter=Beta(name='B_TIME', init=-0.02, fixed=False), variable='time', multiplier=1.0), Term(parameter=Beta(name='B_COST', init=-0.05, fixed=False), variable='cost', multiplier=1.0)])}))

In [3]:
# Translate loading patterns into estimable component scales.
components = [
    ErrorComponent(
        "PUBLIC_TRANSPORT",
        loadings={"TRAIN": 1.0, "SM": 1.0},
        sigma_init=0.35,
    ),
    ErrorComponent(
        "RAIL_CONTRAST",
        loadings={"TRAIN": 1.0, "SM": -1.0},
        sigma_init=0.20,
    ),
]
# Integrate the conditional logits with antithetic normal draws.
model = ErrorComponentsLogit(
    spec,
    components,
    n_draws=32,
    seed=21,
    antithetic=True,
    panel=False,
    device=device,
    max_iter=80,
)
result = model.fit(data)
# Render utility coefficients, component scales, and diagnostics.
display(HTML(result.report().to_html()))


Model family,ErrorComponentsLogit
Estimation objective,Maximum log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,80
Line search,strong_wolfe
